# 00 — Exploring a CAMELS galaxy catalog

Your job in this hackathon is to read a **galaxy catalog** and predict the cosmological
parameters of the universe it came from. Before any of that, you need to know what is
actually in one of these files.

By the end of this notebook you will be able to:

1. open a catalog and see what is inside it,
2. read only the columns you need (this matters — some files are 141 MB),
3. keep the units straight, which is the single most common source of silent bugs,
4. tell a **halo**, a **subhalo** and a **galaxy** apart, and navigate between them,
5. look at a simulation box and at the galaxy mass function of the three suites.

Nothing here is specific to machine learning. It is the domain knowledge you need before
your model choices can be any good.

In [ ]:
# Setup. On Colab this installs the toolkit, mounts the data bucket and points the
# environment variables at it. Anywhere else -- a cluster with the data already on disk --
# it does nothing, which is why there is one set of notebooks rather than two.
import sys

if "google.colab" in sys.modules:
    # --force-reinstall, every time, on purpose. Installing only when the package is
    # missing means anyone who ran a notebook once keeps a stale copy forever, and during
    # an event where fixes are being pushed that is exactly backwards. --no-deps keeps it
    # to a few seconds: everything it depends on is already in the runtime.
    %pip install -q --upgrade --force-reinstall --no-deps git+https://github.com/xwzhang98/kaai-robust-inference-hackathon-2026
    # Drop anything already imported from the old copy, so this works without a restart.
    for _name in [m for m in sys.modules if m.startswith("kaai_hackathon")]:
        del sys.modules[_name]

from kaai_hackathon.colab_setup import setup

setup()


## Setup

The catalogs live wherever `CAMELS_HACKATHON_DATA` points, laid out as

```
$CAMELS_HACKATHON_DATA/<suite>/LH_<n>/groups_090.hdf5
```

with `<suite>` one of `IllustrisTNG`, `SIMBA`, `Astrid`, and `<n>` a simulation id.

In [ ]:
%matplotlib inline
import os
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt

if "CAMELS_HACKATHON_DATA" not in os.environ:
    raise RuntimeError(
        "Set CAMELS_HACKATHON_DATA to the directory holding <suite>/LH_<n>/groups_090.hdf5"
    )
from kaai_hackathon.splits import example_sims

DATA_ROOT = Path(os.environ["CAMELS_HACKATHON_DATA"])
SUITES = ("IllustrisTNG", "SIMBA", "Astrid")


def catalog_path(suite, sim_id):
    return DATA_ROOT / suite / f"LH_{sim_id}" / "groups_090.hdf5"


# A simulation that is certain to exist. The public ids are a pinned random 900 out of
# 1000, not 0..899, so `LH_2` may simply not be in the data you were given.
EXAMPLE = example_sims("IllustrisTNG")[0]

example = catalog_path("IllustrisTNG", EXAMPLE)
print(example, "\nexists:", example.is_file())


## 1. What is inside the file?

A catalog is an HDF5 file. Think of HDF5 as a filesystem inside a single file: it has
**groups** (directories), **datasets** (arrays), and **attributes** (small bits of metadata
attached to either).

Let's look with plain `h5py` first, so you can see the raw structure before we hide it
behind any helper.

In [ ]:
with h5py.File(example, "r") as f:
    print("top level:", list(f.keys()))
    print()
    print("Header attributes:")
    for key in sorted(f["Header"].attrs):
        value = f["Header"].attrs[key]
        print(f"    {key:24s} {value}")

Three of those top-level entries matter:

| entry | what it is |
|---|---|
| `Header` | metadata: box size, redshift, how many objects |
| `Group` | one row per **FoF halo** — a gravitationally bound clump of matter |
| `Subhalo` | one row per **Subfind subhalo** — a bound structure *inside* a halo |

Every dataset inside `Group/` has the same number of rows, and so does every dataset
inside `Subhalo/`. They are two tables.

> **Heads-up about `Header`.** You may have noticed `Omega0` sitting in that list. Yes —
> in a *training* file the header does contain the answer for that simulation. It is not a
> secret and you are welcome to look at it.
>
> But the catalogs you are **scored** on carry a minimal header: `BoxSize`, `Redshift`,
> `HubbleParam`, `NumFiles` and the two object counts, and nothing else. A model that
> reads `Omega0` will therefore score nothing at all. Build on the `Group` and `Subhalo`
> tables, which are complete in every file.

In [ ]:
with h5py.File(example, "r") as f:
    for table in ("Group", "Subhalo"):
        names = sorted(f[table].keys())
        print(f"{table}: {len(names)} columns, {f[table][names[0]].shape[0]} rows")
        for name in names[:6]:
            dataset = f[table][name]
            print(f"    {name:34s} shape={str(dataset.shape):14s} {dataset.dtype}")
        print(f"    ... and {len(names) - 6} more\n")

## 2. Reading a catalog

`h5py` is fine, but two things get tedious fast: remembering to close the file, and
accidentally loading everything. An Astrid catalog is **141 MB across 50 columns**; if you
load all of it for all 900 simulations you will spend your hackathon waiting on disk.

The toolkit gives you three functions and nothing else.

In [ ]:
from kaai_hackathon.catalog_io import read_header, list_fields, read_catalog

# (a) metadata only -- does not touch the tables at all
print(read_header(example))

In [ ]:
# (b) what columns does this file have?
fields = list_fields(example)
print(f"Group:   {len(fields['Group'])} columns")
print(f"Subhalo: {len(fields['Subhalo'])} columns")
print()
print("Subhalo columns:")
for i in range(0, len(fields["Subhalo"]), 3):
    print("   " + "".join(f"{n:38s}" for n in fields["Subhalo"][i:i + 3]))

In [ ]:
# (c) load only what you need -- pass a list of column names
cat = read_catalog(
    example,
    group_fields=["GroupPos", "GroupMass", "GroupNsubs", "GroupFirstSub"],
    subhalo_fields=["SubhaloPos", "SubhaloVel", "SubhaloMass", "SubhaloMassType",
                    "SubhaloGrNr", "SubhaloHalfmassRadType"],
)
print(f"{cat.n_groups} halos, {cat.n_subhalos} subhalos, box = {cat.box_size} ckpc/h")
print("subhalo columns loaded:", sorted(cat.subhalo))
print("SubhaloPos shape:", cat.subhalo["SubhaloPos"].shape)

`read_catalog(path)` with no column list loads everything. Start selective; widen when you
know what you want.

**Which columns should you use?** That is genuinely part of the challenge, and there is no
supplied answer. Some columns describe gravity (positions, velocities, total masses) and
tend to mean the same thing in every simulation code. Others describe gas physics
(star formation, metals, black holes) and are exactly where the three codes disagree most.
Notebook 03 comes back to this.

## 3. Units — read this section twice

CAMELS uses the Gadget convention, which carries factors of the Hubble parameter
$h = 0.6711$ around. Getting this wrong does not raise an error; it just silently makes
your numbers wrong.

| quantity | stored unit | to physical |
|---|---|---|
| positions, radii | ckpc$/h$ | $\times\,10^{-3}/h$ → cMpc |
| masses | $10^{10}\,M_\odot/h$ | $\times\,10^{10}/h$ → $M_\odot$ |
| velocities | km/s | already physical (peculiar) at $z=0$ |

The box is 25000 ckpc$/h$ = **25 cMpc$/h$ ≈ 37 cMpc** on a side. That is small — these are
not big-volume simulations, and that has consequences we will meet later.

In [ ]:
H = 0.6711
BOX_CKPC_H = cat.box_size

mstar_code = cat.subhalo["SubhaloMassType"][:, 4]      # column 4 = stars (see below)
biggest = np.argmax(mstar_code)

print(f"box side          {BOX_CKPC_H:.0f} ckpc/h "
      f"= {BOX_CKPC_H/1000:.1f} cMpc/h = {BOX_CKPC_H/1000/H:.1f} cMpc")
print(f"most massive galaxy: M* = {mstar_code[biggest]:.4f} x 1e10 Msun/h "
      f"= {mstar_code[biggest]*1e10/H:.3e} Msun")
print(f"its position       {cat.subhalo['SubhaloPos'][biggest]} ckpc/h")

### The particle-type axis

Several columns are 2-D with six entries per row. That axis is the **particle type**:

| index | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| type | gas | dark matter | (unused) | (unused) | **stars** | black holes |

So `SubhaloMassType[:, 4]` is stellar mass and `SubhaloMassType[:, 1]` is dark matter mass.
This indexing trips up almost everyone once.

In [ ]:
labels = ["gas", "dark matter", "(unused)", "(unused)", "stars", "black holes"]
totals = cat.subhalo["SubhaloMassType"].sum(axis=0)
for index, (name, total) in enumerate(zip(labels, totals)):
    print(f"  MassType[:, {index}]  {name:12s} total = {total*1e10/H:.3e} Msun")

## 4. Halo, subhalo, galaxy

These three words are not interchangeable.

- A **halo** (a row in `Group/`) is a friends-of-friends clump — the gravitational
  container. Think "a galaxy cluster, or the dark matter halo a galaxy sits in".
- A **subhalo** (a row in `Subhalo/`) is a bound structure found *inside* a halo by the
  Subfind algorithm. A halo always contains at least one subhalo unless it is too diffuse
  to have any.
- A **galaxy** is not a separate table. A galaxy is a subhalo that happens to contain
  stars. You choose the threshold.

The two tables are linked by three columns:

| column | meaning |
|---|---|
| `Subhalo/SubhaloGrNr` | which halo row this subhalo belongs to |
| `Group/GroupFirstSub` | the row index of a halo's **first** subhalo (`-1` if it has none) |
| `Group/GroupNsubs` | how many subhalos that halo has |

Subhalo rows are sorted by halo, so a halo's subhalos are the contiguous block
`GroupFirstSub[g] : GroupFirstSub[g] + GroupNsubs[g]`. The **first** one is the
**central**; the rest are **satellites**.

In [ ]:
nsubs = cat.group["GroupNsubs"]
first = cat.group["GroupFirstSub"]
grnr = cat.subhalo["SubhaloGrNr"]

richest = int(np.argmax(nsubs))
start, count = int(first[richest]), int(nsubs[richest])
rows = np.arange(start, start + count)

print(f"halo {richest} has {count} subhalos, rows {start}..{start+count-1}")
print(f"halo mass {cat.group['GroupMass'][richest]*1e10/H:.3e} Msun\n")
print(f"{'row':>6} {'role':10} {'M_star (Msun)':>15} {'M_total (Msun)':>15}"
      f" {'dist to centre (ckpc/h)':>24}")
centre = cat.subhalo["SubhaloPos"][start]
for i, row in enumerate(rows[:8]):
    offset = cat.subhalo["SubhaloPos"][row] - centre
    offset -= cat.box_size * np.round(offset / cat.box_size)     # periodic!
    print(f"{row:6d} {'central' if i == 0 else 'satellite':10}"
          f" {cat.subhalo['SubhaloMassType'][row,4]*1e10/H:15.3e}"
          f" {cat.subhalo['SubhaloMass'][row]*1e10/H:15.3e}"
          f" {np.linalg.norm(offset):24.1f}")
if count > 8:
    print(f"{'':6} ... and {count-8} more satellites")

Notice the `-= box * round(delta / box)` in that distance calculation. **The box is
periodic**: a galaxy at x = 24900 and one at x = 100 are 200 ckpc/h apart, not 24800.
Any distance you compute has to use this "minimum image" convention. Forgetting it is the
second most common bug in this dataset, after units.

Here is the flag telling you whether each subhalo is a central, built from the linkage:

In [ ]:
is_central = np.zeros(cat.n_subhalos, dtype=bool)
is_central[first[nsubs > 0]] = True

print(f"centrals   {is_central.sum():6d}")
print(f"satellites {(~is_central).sum():6d}")

# sanity: every subhalo's group agrees with the block it sits in
assert np.all(grnr[first[nsubs > 0]] == np.flatnonzero(nsubs > 0))
print("\nlinkage is self-consistent")

### From subhalos to galaxies

Most subhalos have no stars at all — they are pure dark matter clumps. And of those that do
have stars, the ones near the resolution limit are made of only a handful of star
particles, so their properties are noise.

A common choice is to keep subhalos with $M_\star > 1.3\times10^{8}\,M_\odot/h$, which is
roughly 20 star particles at CAMELS resolution. **This threshold is a modelling choice you
are free to change**, and it matters: it sets how many objects your model sees.

In [ ]:
MSTAR_CUT = 1.3e-2      # in code units of 1e10 Msun/h  ->  1.3e8 Msun/h

for suite in SUITES:
    c = read_catalog(catalog_path(suite, example_sims(suite)[0]), group_fields=["GroupNsubs"],
                     subhalo_fields=["SubhaloMassType"])
    mstar = c.subhalo["SubhaloMassType"][:, 4]
    print(f"{suite:14s} {c.n_subhalos:6d} subhalos | "
          f"{int((mstar > 0).sum()):6d} with any stars | "
          f"{int((mstar > MSTAR_CUT).sum()):5d} above the cut")

Those numbers already tell you something important: **the three suites do not agree**.
Same box size, same resolution, same kind of object — very different galaxy counts. That
disagreement is the distribution shift this hackathon is about.

## 5. Look at the box

Let's plot the galaxies of one simulation, projected along $z$, with marker size following
stellar mass.

In [ ]:
c = read_catalog(catalog_path("IllustrisTNG", EXAMPLE),
                 group_fields=[], subhalo_fields=["SubhaloPos", "SubhaloMassType"])
mstar = c.subhalo["SubhaloMassType"][:, 4]
keep = mstar > MSTAR_CUT
pos = c.subhalo["SubhaloPos"][keep] / 1000.0          # -> cMpc/h
size = 3 + 40 * (np.log10(mstar[keep]) - np.log10(MSTAR_CUT)) / 3.0

fig, ax = plt.subplots(figsize=(6.5, 6.2))
ax.scatter(pos[:, 0], pos[:, 1], s=np.clip(size, 3, 90),
           c=np.log10(mstar[keep] * 1e10 / 0.6711), cmap="viridis",
           alpha=0.85, edgecolor="none")
ax.set_xlim(0, c.box_size / 1000); ax.set_ylim(0, c.box_size / 1000)
ax.set_xlabel("x  [cMpc/h]"); ax.set_ylabel("y  [cMpc/h]")
ax.set_title(f"IllustrisTNG LH_0 — {keep.sum()} galaxies above the cut")
ax.set_aspect("equal")
fig.colorbar(ax.collections[0], ax=ax, label=r"$\log_{10}(M_\star/M_\odot)$", shrink=0.82)
plt.tight_layout(); plt.show()

Galaxies are not scattered at random: they sit in filaments and knots. That structure is
the **cosmic web**, and how clumpy it is depends on the cosmological parameters — which is
precisely why the parameters are recoverable from a catalog at all.

The edges wrap around. A structure running off the right edge continues at the left.

## 6. The stellar mass function

The mass function — how many galaxies there are at each mass — is the most basic summary
of a catalog and a reasonable first feature.

One subtlety before we plot it. Every simulation in this dataset has **different
parameters**: $\Omega_m$ ranges from about 0.1 to 0.5, and the feedback strengths vary by
more than an order of magnitude. So if you plot individual simulations, the scatter is
enormous — and that scatter *is the signal you are trying to learn*, not noise.

To compare the three codes we therefore average over many simulations and show the
uncertainty **on the mean** (the standard error), which is what tells you whether the
codes really sit at different levels.

In [ ]:
BINS = np.linspace(8.0, 12.0, 25)
CENTRES = 0.5 * (BINS[1:] + BINS[:-1])
N_SIMS_AVG = 20

curves = {}
for suite in SUITES:
    stacked = []
    for sim_id in example_sims(suite, N_SIMS_AVG):
        c = read_catalog(catalog_path(suite, sim_id), group_fields=[],
                         subhalo_fields=["SubhaloMassType"])
        mstar = c.subhalo["SubhaloMassType"][:, 4]
        mstar = mstar[mstar > 0] * 1e10 / 0.6711
        stacked.append(np.histogram(np.log10(mstar), bins=BINS)[0])
    curves[suite] = np.asarray(stacked, dtype=float)
    print(f"{suite:14s} averaged {N_SIMS_AVG} simulations")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))

# left: mean per suite, band = standard error ON THE MEAN
ax = axes[0]
for suite in SUITES:
    stacked = curves[suite]
    mean = stacked.mean(axis=0)
    sem = stacked.std(axis=0, ddof=1) / np.sqrt(stacked.shape[0])
    ax.plot(CENTRES, mean, marker="o", ms=3.5, label=suite)
    ax.fill_between(CENTRES, np.maximum(mean - sem, 1e-2), mean + sem, alpha=0.3)
ax.axvline(np.log10(1.3e8 / 0.6711), ls="--", c="0.4", lw=1)
ax.set_yscale("log"); ax.set_ylim(0.5, 1e3)
ax.set_xlabel(r"$\log_{10}(M_\star / M_\odot)$")
ax.set_ylabel("galaxies per bin")
ax.set_title(f"Mean over {N_SIMS_AVG} sims, band = error on the mean")
ax.legend(loc="upper right", fontsize=9)
ax.annotate("resolution cut", xy=(np.log10(1.3e8 / 0.6711), 0.9), xytext=(8.45, 0.75),
            fontsize=9, color="0.4",
            arrowprops=dict(arrowstyle="->", color="0.4", lw=0.8))

# right: the individual simulations behind one of those means
ax = axes[1]
for row in curves["IllustrisTNG"]:
    ax.plot(CENTRES, np.maximum(row, 1e-2), color="C0", alpha=0.25, lw=1)
ax.plot(CENTRES, curves["IllustrisTNG"].mean(axis=0), color="k", lw=2, label="mean")
ax.set_yscale("log"); ax.set_ylim(0.5, 1e3)
ax.set_xlabel(r"$\log_{10}(M_\star / M_\odot)$")
ax.set_title(f"IllustrisTNG: the {N_SIMS_AVG} individual simulations")
ax.legend(loc="upper right", fontsize=9)

plt.tight_layout(); plt.show()

Two different messages in those two panels, and you need both.

**Right panel** — within a *single* suite, individual simulations differ by more than an
order of magnitude at fixed mass. That is not noise: it is $\Omega_m$, $\sigma_8$ and the
feedback parameters doing their work. This spread is the thing your model reads to infer
the parameters. If it were small, the task would be impossible.

**Left panel** — average that spread away and the three codes still land at systematically
different levels, well outside the error on the mean. All three simulate the same physics,
in the same box, at the same resolution. They disagree because they make different choices
about how supernovae and black holes push gas around.

Put together: a model that learns "this many galaxies of this mass implies these
parameters" from IllustrisTNG alone has learned a mapping that is *offset* on SIMBA. It
will be confidently wrong. Building something that survives moving between codes is the
point of this hackathon.

## 7. Not every suite has every column

One last practical thing. The suites do not all ship the same columns.

In [ ]:
OOD_ONLY_MISSING = ("SubhaloStellarPhotometrics",
                    "SubhaloStellarPhotometricsMassInRad",
                    "SubhaloStellarPhotometricsRad")

for suite in SUITES:
    f = list_fields(catalog_path(suite, example_sims(suite)[0]))
    print(f"{suite:14s} Group={len(f['Group'])}  Subhalo={len(f['Subhalo'])}")

print("\nPresent in all three suites you can train on, but ABSENT from the held-out"
      "\nout-of-distribution suite:")
for name in OOD_ONLY_MISSING:
    here = name in list_fields(catalog_path("IllustrisTNG", EXAMPLE))["Subhalo"]
    print(f"    {name:38s} here={here}   OOD suite=False")
print("\nSo the columns guaranteed everywhere are 25 Group + 47 Subhalo.")

The three suites you can train on all ship the same 25 + 50 columns, so within your
training data there is nothing to worry about.

The catch is the **held-out out-of-distribution suite** — a fourth simulation code you
never see, which is part of how you are scored. It has the same 25 `Group` columns but only
**47** `Subhalo` columns: the three `SubhaloStellarPhotometrics*` entries above do not
exist there.

So the set of columns guaranteed to exist everywhere is **25 Group + 47 Subhalo**. A model
built on those three photometry columns will break, or silently degrade, exactly when it is
being tested hardest. Call `list_fields` and handle what is missing rather than assuming.

## What you should take away

1. A catalog is two tables — halos and subhalos — linked by `SubhaloGrNr`,
   `GroupFirstSub` and `GroupNsubs`.
2. A galaxy is a subhalo with stars, above a threshold you choose.
3. Positions are ckpc$/h$, masses are $10^{10} M_\odot/h$, and the box is periodic. Every
   distance needs the minimum-image convention.
4. `MassType[:, 4]` is stars, `[:, 1]` is dark matter.
5. Load only the columns you need.
6. The three suites genuinely disagree, and that disagreement is the task.

**Next:** notebook `01` introduces the labels — the six parameters you are predicting — and
what each of them does to a catalog.